# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |


In [1]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

#@markdown **Choose your TTS (voice synthesis) engine:**
tts_engine = "Qwen (recommended)" #@param ["Qwen (recommended)", "Chatterbox"]

import subprocess, sys, shutil, time

# ── Python packages ──────────────────────────────────────────────
if "Chatterbox" in tts_engine:
    extras = "tts-chatterbox,transcribe-faster"
    print("Installing Mazinger with Chatterbox TTS + Faster Whisper...")
else:
    extras = "tts,transcribe-faster"
    print("Installing Mazinger with Qwen TTS + Faster Whisper...")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        f"mazinger[{extras}]", "gradio>=4.0"])

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        stdout=subprocess.DEVNULL,
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("✅ Ollama server started (model will be pulled on first run)")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n✅ Ready! Run the next cell to download & prepare models.")


Installing Mazinger with Qwen TTS + Faster Whisper...



📥 Installing Ollama (local LLM server)...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
############################################################################################################################################################################### 100.0%
>>> Creating ollama user...
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


✅ Ollama installed
✅ Ollama server started (model will be pulled on first run)

✅ GPU detected: NVIDIA GeForce RTX 4060 Ti

✅ Ready! Run the next cell to download & prepare models.


In [2]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster-Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS / Chatterbox: ~3-4 GB

import subprocess, sys, os, time

print("📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print("1️⃣  Pulling Ollama LLM model (qwen3.5:2b-q8_0)...")
try:
    result = subprocess.run(
        ["ollama", "pull", "qwen3.5:2b-q8_0"],
        timeout=600,
    )
    if result.returncode == 0:
        print("   ✅ Ollama model ready\n")
    else:
        print("   ⚠️  Ollama pull had issues (model will be pulled on first run)\n")
except Exception as e:
    print(f"   ⚠️  Ollama pull failed: {e}\n")

# ── 2. Faster-Whisper model ─────────────────────────────────────
print("2️⃣  Downloading Faster-Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster-Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster-Whisper download failed: {e}\n")

# ── 3. TTS model ────────────────────────────────────────────────
# `tts_engine` was set by Step 1
if "Chatterbox" in tts_engine:
    print("3️⃣  Downloading Chatterbox TTS model...")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("ResembleAI/chatterbox")
        print("   ✅ Chatterbox model ready\n")
    except Exception as e:
        print(f"   ⚠️  Chatterbox download failed: {e}\n")
else:
    print("3️⃣  Downloading Qwen TTS model...")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
        print("   ✅ Qwen TTS model ready\n")
    except Exception as e:
        print(f"   ⚠️  Qwen TTS download failed: {e}\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

📥 Downloading models (this may take 5-10 min on first run)...

1️⃣  Pulling Ollama LLM model (qwen3.5:2b-q8_0)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling b709d81508a0:   0% ▕                  ▏ 3.4 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:   5% ▕                  ▏ 124 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:   7% ▕█                 ▏ 179 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:  11% ▕██                ▏ 311 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:  17% ▕██                ▏ 455 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:  18% ▕███               ▏ 501 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:  23% ▕████              ▏ 628 MB/2.7 GB                  pulling manifest 
pulling b709d81508a0:  28% ▕████              ▏ 761 MB/2.7 GB                  pulling manifest 
pulling b709d815

   ✅ Ollama model ready

2️⃣  Downloading Faster-Whisper model (large-v3)...


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

...

   ✅ Faster-Whisper model ready

3️⃣  Downloading Qwen TTS model...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

   ✅ Qwen TTS model ready

✅ All models downloaded and cached! Run the next cell to launch the app.


In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import gradio as gr
import logging
import os
import subprocess as sp
import threading
import time
import traceback


# ═══════════════════════════════════════════════════════════════════════
#  Log capture — streams pipeline logs to the Gradio UI in real-time
# ═══════════════════════════════════════════════════════════════════════

class _LogCollector(logging.Handler):
    """Thread-safe log handler that buffers formatted messages."""

    def __init__(self):
        super().__init__()
        self._lines: list[str] = []
        self._lock = threading.Lock()

    def emit(self, record):
        with self._lock:
            self._lines.append(self.format(record))

    def read(self) -> str:
        with self._lock:
            return "\n".join(self._lines)

    def clear(self):
        with self._lock:
            self._lines.clear()


# ═══════════════════════════════════════════════════════════════════════
#  Ollama helper — ensure server is running and model is pulled
# ═══════════════════════════════════════════════════════════════════════

_OLLAMA_DEFAULT_MODEL = "qwen3.5:2b-q8_0"


def _ensure_ollama(model_id: str | None = None):
    """Start Ollama server (if needed) and pull the requested model."""
    model_id = model_id or _OLLAMA_DEFAULT_MODEL

    # Start server if not already running
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
    except Exception:
        sp.Popen(["ollama", "serve"], stdout=sp.DEVNULL, stderr=sp.DEVNULL)
        time.sleep(3)

    # Check if model is already available
    result = sp.run(["ollama", "list"], capture_output=True, text=True, timeout=10)
    for line in result.stdout.strip().splitlines()[1:]:
        if line.split()[0] == model_id:
            return  # already pulled

    # Pull the model
    sp.run(["ollama", "pull", model_id], check=True, timeout=600)


# ═══════════════════════════════════════════════════════════════════════
#  Constants
# ═══════════════════════════════════════════════════════════════════════

LANGUAGES = [
    "Arabic", "Bengali", "Chinese (Simplified)", "Chinese (Traditional)",
    "Czech", "Danish", "Dutch", "English", "Finnish", "French",
    "German", "Greek", "Hebrew", "Hindi", "Hungarian",
    "Indonesian", "Italian", "Japanese", "Korean", "Malay",
    "Norwegian", "Persian", "Polish", "Portuguese",
    "Romanian", "Russian", "Spanish", "Swedish",
    "Thai", "Turkish", "Ukrainian", "Urdu", "Vietnamese",
]

VOICE_PRESETS = ["abubakr", "daheeh-v1", "italian-v1", "trump-v1"]

QUALITY_MAP = {"Low (360p)": "low", "Medium (720p)": "medium", "High (best)": "high"}
METHOD_MAP = {
    "OpenAI Whisper (cloud)": "openai",
    "Faster Whisper (local GPU)": "faster-whisper",
    "WhisperX (local GPU)": "whisperx",
}


# ═══════════════════════════════════════════════════════════════════════
#  Custom Gradio Theme — dark slate + blue/violet accents
# ═══════════════════════════════════════════════════════════════════════

theme = gr.themes.Base(
    primary_hue=gr.themes.Color(
        c50="#eff6ff", c100="#dbeafe", c200="#bfdbfe", c300="#93c5fd",
        c400="#60a5fa", c500="#3b82f6", c600="#2563eb", c700="#1d4ed8",
        c800="#1e40af", c900="#1e3a8a", c950="#172554",
    ),
    secondary_hue=gr.themes.Color(
        c50="#f5f3ff", c100="#ede9fe", c200="#ddd6fe", c300="#c4b5fd",
        c400="#a78bfa", c500="#8b5cf6", c600="#7c3aed", c700="#6d28d9",
        c800="#5b21b6", c900="#4c1d95", c950="#2e1065",
    ),
    neutral_hue=gr.themes.Color(
        c50="#f8fafc", c100="#f1f5f9", c200="#e2e8f0", c300="#cbd5e1",
        c400="#94a3b8", c500="#64748b", c600="#475569", c700="#334155",
        c800="#1e293b", c900="#0f172a", c950="#020617",
    ),
    font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "monospace"],
).set(
    body_background_fill="#0f172a",
    body_background_fill_dark="#0f172a",
    body_text_color="#e2e8f0",
    body_text_color_dark="#e2e8f0",
    body_text_color_subdued="#94a3b8",
    body_text_color_subdued_dark="#94a3b8",

    block_background_fill="#1e293b",
    block_background_fill_dark="#1e293b",
    block_border_color="#334155",
    block_border_color_dark="#334155",
    block_border_width="1px",
    block_label_text_color="#94a3b8",
    block_label_text_color_dark="#94a3b8",
    block_label_background_fill="#1e293b",
    block_label_background_fill_dark="#1e293b",
    block_title_text_color="#e2e8f0",
    block_title_text_color_dark="#e2e8f0",
    block_shadow="0 4px 6px -1px rgba(0, 0, 0, 0.3)",
    block_shadow_dark="0 4px 6px -1px rgba(0, 0, 0, 0.3)",

    input_background_fill="#0f172a",
    input_background_fill_dark="#0f172a",
    input_border_color="#334155",
    input_border_color_dark="#334155",
    input_placeholder_color="#475569",
    input_placeholder_color_dark="#475569",

    button_primary_background_fill="linear-gradient(135deg, #3b82f6 0%, #8b5cf6 100%)",
    button_primary_background_fill_dark="linear-gradient(135deg, #3b82f6 0%, #8b5cf6 100%)",
    button_primary_background_fill_hover="linear-gradient(135deg, #2563eb 0%, #7c3aed 100%)",
    button_primary_background_fill_hover_dark="linear-gradient(135deg, #2563eb 0%, #7c3aed 100%)",
    button_primary_text_color="#ffffff",
    button_primary_border_color="transparent",
    button_primary_border_color_dark="transparent",

    button_secondary_background_fill="#334155",
    button_secondary_background_fill_dark="#334155",
    button_secondary_text_color="#e2e8f0",
    button_secondary_border_color="#475569",

    border_color_primary="#3b82f6",
    border_color_primary_dark="#3b82f6",

    shadow_spread="0px",

    checkbox_background_color="#0f172a",
    checkbox_background_color_dark="#0f172a",
    checkbox_background_color_selected="#3b82f6",
    checkbox_background_color_selected_dark="#3b82f6",
    checkbox_border_color="#475569",
    checkbox_border_color_dark="#475569",
    checkbox_label_text_color="#e2e8f0",

    slider_color="#3b82f6",
    slider_color_dark="#3b82f6",
)


# ═══════════════════════════════════════════════════════════════════════
#  Custom CSS
# ═══════════════════════════════════════════════════════════════════════

CSS = """
/* ── Global layout ──────────────────────────────────────────────── */
.gradio-container {
    max-width: 900px !important;
    margin: 0 auto !important;
    background: #0f172a !important;
}
footer { display: none !important; }

/* ── Header ─────────────────────────────────────────────────────── */
.app-header {
    text-align: center;
    padding: 1.5rem 1rem 0.5rem;
}
.app-header h1 {
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    background: linear-gradient(135deg, #60a5fa, #a78bfa, #f472b6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 0.25rem !important;
}
.app-header p {
    color: #94a3b8 !important;
    font-size: 1rem;
    margin-top: 0 !important;
}

/* ── Section headings ───────────────────────────────────────────── */
.section-title {
    color: #e2e8f0 !important;
    font-size: 0.85rem !important;
    font-weight: 600 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.08em !important;
    margin: 0.5rem 0 0.1rem !important;
    padding: 0 !important;
}

/* ── Cards ──────────────────────────────────────────────────────── */
.card {
    background: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    padding: 1rem !important;
}
.card-highlight {
    background: linear-gradient(135deg, rgba(59,130,246,0.08), rgba(139,92,246,0.08)) !important;
    border: 1px solid rgba(59,130,246,0.25) !important;
    border-radius: 12px !important;
    padding: 1rem !important;
}

/* ── Inputs ─────────────────────────────────────────────────────── */
.gradio-container input[type="text"],
.gradio-container input[type="password"],
.gradio-container textarea,
.gradio-container select {
    background: #0f172a !important;
    border: 1px solid #334155 !important;
    border-radius: 8px !important;
    color: #e2e8f0 !important;
    transition: border-color 0.2s ease;
}
.gradio-container input:focus,
.gradio-container textarea:focus,
.gradio-container select:focus {
    border-color: #3b82f6 !important;
    box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.15) !important;
}

/* ── Primary button ─────────────────────────────────────────────── */
.run-btn {
    font-size: 1.1rem !important;
    font-weight: 700 !important;
    padding: 0.85rem 2rem !important;
    border-radius: 12px !important;
    letter-spacing: 0.02em;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
    box-shadow: 0 4px 15px rgba(59, 130, 246, 0.3) !important;
}
.run-btn:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 20px rgba(59, 130, 246, 0.4) !important;
}

/* ── Log area ───────────────────────────────────────────────────── */
.log-box textarea {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.78rem !important;
    line-height: 1.5 !important;
    background: #020617 !important;
    color: #94a3b8 !important;
    border-radius: 8px !important;
}

/* ── Results area ───────────────────────────────────────────────── */
.results-card {
    background: linear-gradient(135deg, rgba(59,130,246,0.06), rgba(139,92,246,0.06)) !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    padding: 1rem !important;
}

/* ── Accordion ──────────────────────────────────────────────────── */
.gradio-accordion {
    background: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
}
.gradio-accordion > .label-wrap {
    background: #1e293b !important;
    color: #94a3b8 !important;
}

/* ── Tabs inside accordion ──────────────────────────────────────── */
.gradio-tab-nav button {
    color: #94a3b8 !important;
    font-size: 0.82rem !important;
    border: none !important;
    background: transparent !important;
}
.gradio-tab-nav button.selected {
    color: #60a5fa !important;
    border-bottom: 2px solid #3b82f6 !important;
}

/* ── Divider ────────────────────────────────────────────────────── */
.divider {
    border: none !important;
    border-top: 1px solid #1e293b !important;
    margin: 0.5rem 0 !important;
}

/* ── File upload area ───────────────────────────────────────────── */
.gradio-file {
    border: 2px dashed #334155 !important;
    border-radius: 12px !important;
    background: rgba(15, 23, 42, 0.5) !important;
}

/* ── Dropdown list (options popup) ──────────────────────────────── */
.gradio-container ul[role="listbox"],
.gradio-container .options,
.gradio-container ul.options {
    background: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 8px !important;
}
.gradio-container ul[role="listbox"] li,
.gradio-container .options li,
.gradio-container ul.options li {
    color: #e2e8f0 !important;
    background: #1e293b !important;
}
.gradio-container ul[role="listbox"] li:hover,
.gradio-container .options li:hover,
.gradio-container ul.options li:hover,
.gradio-container ul[role="listbox"] li.selected,
.gradio-container .options li.selected {
    background: #334155 !important;
    color: #ffffff !important;
}

/* ── Ollama / OpenAI info text ──────────────────────────────────── */
.ollama-info p {
    color: #86efac !important;
    font-size: 0.85rem !important;
    margin: 0.3rem 0 0 !important;
}
.openai-info p {
    color: #94a3b8 !important;
    font-size: 0.85rem !important;
    margin: 0.3rem 0 0 !important;
}

/* ── Cookie guide ───────────────────────────────────────────────── */
.cookie-guide-step {
    background: #0f172a !important;
    border: 1px solid #334155 !important;
    border-radius: 10px !important;
    padding: 0.8rem !important;
    margin-bottom: 0.6rem !important;
}
.cookie-guide-step p {
    color: #cbd5e1 !important;
    font-size: 0.85rem !important;
    margin: 0.4rem 0 0.5rem !important;
    line-height: 1.5 !important;
}
.cookie-guide-step img {
    border-radius: 8px !important;
    border: 1px solid #334155 !important;
    max-width: 100% !important;
}
.cookie-guide-step a {
    color: #60a5fa !important;
    text-decoration: none !important;
}
.cookie-guide-step a:hover {
    text-decoration: underline !important;
}
.cookie-step-num {
    display: inline-block;
    background: linear-gradient(135deg, #3b82f6, #8b5cf6);
    color: #fff !important;
    font-weight: 700;
    width: 1.5rem;
    height: 1.5rem;
    line-height: 1.5rem;
    text-align: center;
    border-radius: 50%;
    font-size: 0.8rem;
    margin-right: 0.4rem;
}
"""


# ═══════════════════════════════════════════════════════════════════════
#  Pipeline runner
# ═══════════════════════════════════════════════════════════════════════

def run_dubbing(
    source_type, url, uploaded_file,
    cookies_text,
    target_language, voice_type, voice_preset, voice_file, voice_script_text,
    llm_provider, ollama_model, openai_key,
    api_base_url, llm_model,
    quality, start_time, end_time,
    transcribe_method, whisper_model,
    source_language, words_per_second, duration_budget, translate_technical,
    tts_engine, tts_dtype, chatterbox_exaggeration, chatterbox_cfg,
    tempo_mode, max_tempo, loudness_match, mix_background, background_volume,
    output_type, force_reset,
):
    """Generator → yields (status, logs, audio, video) tuples."""

    is_ollama = (llm_provider == "Ollama (Local — Free)")

    if not is_ollama and (not openai_key or not openai_key.strip()):
        yield "❌ Please enter your OpenAI API key.", "", None, None
        return

    # Ensure Ollama server + model are ready
    if is_ollama:
        yield "⏳ Checking Ollama server and model…", "", None, None
        try:
            _ensure_ollama(ollama_model.strip() if ollama_model else None)
        except Exception as exc:
            yield f"❌ Ollama setup failed: {exc}", "", None, None
            return

    source = None
    if source_type == "YouTube URL":
        if not url or not url.strip():
            yield "❌ Please enter a video URL.", "", None, None
            return
        source = url.strip()
    else:
        if not uploaded_file:
            yield "❌ Please upload a video or audio file.", "", None, None
            return
        source = uploaded_file

    if voice_type == "Preset Voice":
        if not voice_preset:
            yield "❌ Please select a voice preset.", "", None, None
            return
    else:
        if not voice_file:
            yield "❌ Please upload a voice sample (10-30 sec audio clip).", "", None, None
            return
        if not voice_script_text or not voice_script_text.strip():
            yield "❌ Please enter the transcript of your voice sample.", "", None, None
            return

    collector = _LogCollector()
    collector.setFormatter(logging.Formatter(
        "%(asctime)s  %(message)s", datefmt="%H:%M:%S"
    ))
    maz_log = logging.getLogger("mazinger")
    maz_log.setLevel(logging.INFO)
    maz_log.addHandler(collector)

    yield "⏳ Preparing voice profile…", "", None, None

    try:
        if voice_type == "Preset Voice":
            from mazinger.profiles import fetch_profile
            voice_sample_path, voice_script_path = fetch_profile(voice_preset)
        else:
            voice_sample_path = voice_file
            voice_script_path = voice_script_text.strip()
    except Exception as exc:
        maz_log.removeHandler(collector)
        yield f"❌ Voice profile error: {exc}", collector.read(), None, None
        return

    result = {}
    error_box = {}
    done = threading.Event()

    def _worker():
        try:
            from mazinger import MazingerDubber

            # Resolve API credentials based on LLM provider choice
            if is_ollama:
                _api_key = "ollama"
                _base_url = "http://localhost:11434/v1"
                _llm = (ollama_model.strip()
                        if ollama_model and ollama_model.strip()
                        else _OLLAMA_DEFAULT_MODEL)
            else:
                _api_key = openai_key.strip()
                _base_url = (api_base_url.strip()
                             if api_base_url and api_base_url.strip() else None)
                _llm = (llm_model.strip()
                        if llm_model and llm_model.strip() else None)

            os.environ["OPENAI_API_KEY"] = _api_key

            init_kw = dict(openai_api_key=_api_key)
            if _base_url:
                init_kw["openai_base_url"] = _base_url
            if _llm:
                init_kw["llm_model"] = _llm

            dubber = MazingerDubber(**init_kw)

            try:
                import torch
                device = "cuda" if torch.cuda.is_available() else "cpu"
            except ImportError:
                device = "cpu"

            # Write cookies to a temp file if provided
            _cookies_path = None
            if cookies_text and cookies_text.strip():
                import tempfile
                _cookies_path = os.path.join(tempfile.gettempdir(), "mazinger_cookies.txt")
                with open(_cookies_path, "w", encoding="utf-8") as _cf:
                    _cf.write(cookies_text.strip())

            dub_kw = dict(
                source=source,
                voice_sample=voice_sample_path,
                voice_script=voice_script_path,
                device=device,
                target_language=target_language,
                output_type=output_type.lower(),
                force_reset=force_reset,
                tts_engine=tts_engine.lower(),
                tts_dtype=tts_dtype,
                tempo_mode=tempo_mode.lower(),
                max_tempo=max_tempo,
                loudness_match=loudness_match,
                mix_background=mix_background,
                background_volume=background_volume,
                translate_technical_terms=translate_technical,
                **(dict(cookies=_cookies_path) if _cookies_path else {}),
            )

            if source_language and source_language != "Auto-detect":
                dub_kw["source_language"] = source_language
            q = QUALITY_MAP.get(quality)
            if q:
                dub_kw["quality"] = q
            if start_time and start_time.strip():
                dub_kw["start"] = start_time.strip()
            if end_time and end_time.strip():
                dub_kw["end"] = end_time.strip()
            m = METHOD_MAP.get(transcribe_method)
            # Ollama doesn't support OpenAI Whisper — force local transcription
            if is_ollama and m == "openai":
                m = "faster-whisper"
            if m:
                dub_kw["transcribe_method"] = m
            if whisper_model and whisper_model.strip():
                dub_kw["whisper_model"] = whisper_model.strip()
            if words_per_second != 2.0:
                dub_kw["words_per_second"] = words_per_second
            if duration_budget != 0.80:
                dub_kw["duration_budget"] = duration_budget
            if tts_engine.lower() == "chatterbox":
                dub_kw["chatterbox_exaggeration"] = chatterbox_exaggeration
                dub_kw["chatterbox_cfg"] = chatterbox_cfg

            paths = dubber.dub(**dub_kw)
            result["paths"] = paths

        except Exception as exc:
            error_box["error"] = exc
            logging.getLogger("mazinger").error(
                "Pipeline failed: %s\n%s", exc, traceback.format_exc()
            )
        finally:
            done.set()

    thread = threading.Thread(target=_worker, daemon=True)
    thread.start()

    while not done.is_set():
        time.sleep(2)
        yield (
            "⏳ Processing… (this may take several minutes)",
            collector.read(),
            None,
            None,
        )

    maz_log.removeHandler(collector)

    if "error" in error_box:
        yield (
            f"❌ Pipeline failed: {error_box['error']}",
            collector.read(),
            None,
            None,
        )
        return

    paths = result.get("paths")
    audio_out = None
    video_out = None

    if paths:
        if hasattr(paths, "final_audio") and os.path.isfile(paths.final_audio):
            audio_out = paths.final_audio
        if hasattr(paths, "final_video") and os.path.isfile(paths.final_video):
            video_out = paths.final_video

    status_parts = ["✅ Dubbing complete!"]
    if audio_out:
        status_parts.append(f"Audio → {audio_out}")
    if video_out:
        status_parts.append(f"Video → {video_out}")

    yield "\n".join(status_parts), collector.read(), audio_out, video_out


# ═══════════════════════════════════════════════════════════════════════
#  Build the Gradio interface
# ═══════════════════════════════════════════════════════════════════════

with gr.Blocks(theme=theme, title="Mazinger Studio", css=CSS) as app:

    # ── Header ────────────────────────────────────────────────────
    gr.Markdown(
        "# 🎬 Mazinger Studio\n"
        "Dub any video into another language with AI — paste a URL, pick a voice, and go.",
        elem_classes="app-header",
    )

    # ── LLM Provider ──────────────────────────────────────────────
    gr.Markdown("#### 🤖  LLM PROVIDER", elem_classes="section-title")
    with gr.Group(elem_classes="card-highlight"):
        llm_provider = gr.Radio(
            ["Ollama (Local — Free)", "OpenAI (Cloud)"],
            value="Ollama (Local — Free)",
            label="Translation engine",
            container=False,
        )

        with gr.Group(visible=True) as ollama_group:
            ollama_model = gr.Textbox(
                label="Ollama Model",
                value=_OLLAMA_DEFAULT_MODEL,
                placeholder="e.g. qwen3.5:2b-q8_0, llama3.1:8b, …",
                info="Model will be pulled automatically on first run",
            )
            gr.Markdown(
                "✅ **No API key needed.** Runs 100% locally.  \n"
                "Transcription uses local Faster Whisper on your GPU.",
                elem_classes="ollama-info",
            )

        with gr.Group(visible=False) as openai_group:
            openai_key = gr.Textbox(
                label="OpenAI API Key",
                type="password",
                placeholder="sk-…",
                info="Required for transcription (Whisper) and translation (GPT)",
            )
            gr.Markdown(
                "Uses OpenAI Whisper for transcription and GPT for translation.",
                elem_classes="openai-info",
            )

    gr.HTML('<hr class="divider">')

    # ── Source ─────────────────────────────────────────────────────
    gr.Markdown("#### 📹  SOURCE", elem_classes="section-title")
    with gr.Group(elem_classes="card"):
        source_type = gr.Radio(
            ["YouTube URL", "Upload File"],
            value="YouTube URL",
            label="Source type",
            container=False,
        )
        url_input = gr.Textbox(
            label="Video URL",
            placeholder="https://www.youtube.com/watch?v=…",
            visible=True,
        )
        file_input = gr.File(
            label="Upload a video or audio file",
            file_types=[".mp4", ".mkv", ".webm", ".mov", ".mp3", ".wav", ".m4a"],
            visible=False,
        )

    def _toggle_source(choice):
        return (
            gr.update(visible=(choice == "YouTube URL")),
            gr.update(visible=(choice == "Upload File")),
        )
    source_type.change(_toggle_source, source_type, [url_input, file_input])

    # ── YouTube Cookies (collapsed) ───────────────────────────────
    _IMG_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/assets/yt-cache"
    _EXT_URL = "https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc"

    with gr.Accordion(
        "🍪  YouTube Cookies (only if downloads fail)",
        open=False,
    ):
        gr.Markdown(
            "Some YouTube videos require authentication to download. "
            "If you see a download error, paste your YouTube cookies below.\n\n"
            "*Don't know how? Click **How to get cookies** below.*",
            elem_classes="openai-info",
        )
        cookies_text = gr.Textbox(
            label="Cookies (Netscape format)",
            placeholder="# Netscape HTTP Cookie File\n# Paste your cookies here…",
            lines=4,
            max_lines=12,
        )
        with gr.Accordion("📖  How to get cookies", open=False):
            gr.HTML(
                '<div class="cookie-guide-step">'
                '<p><span class="cookie-step-num">1</span> '
                f'Install the <a href="{_EXT_URL}" target="_blank">'
                'Get cookies.txt locally</a> Chrome extension</p>'
                f'<img src="{_IMG_BASE}/p1.png" alt="Step 1: Install the Chrome extension" />'
                '</div>'
                '<div class="cookie-guide-step">'
                '<p><span class="cookie-step-num">2</span> '
                'Go to <a href="https://www.youtube.com" target="_blank">youtube.com</a>, '
                'make sure you are logged in, then click the extension icon</p>'
                f'<img src="{_IMG_BASE}/p2.png" alt="Step 2: Open extension on YouTube" />'
                '</div>'
                '<div class="cookie-guide-step">'
                '<p><span class="cookie-step-num">3</span> '
                'Click <strong>Copy</strong> to copy the cookies, '
                'then paste them in the text box above</p>'
                f'<img src="{_IMG_BASE}/p3.png" alt="Step 3: Copy cookies" />'
                '</div>'
            )

    # ── Voice & Language ──────────────────────────────────────────
    gr.Markdown("#### 🎤  VOICE & LANGUAGE", elem_classes="section-title")
    with gr.Group(elem_classes="card"):
        with gr.Row(equal_height=True):
            target_language = gr.Dropdown(
                choices=LANGUAGES,
                value="English",
                label="Output language",
                scale=1,
            )
            voice_type = gr.Radio(
                ["Preset Voice", "Custom Voice"],
                value="Preset Voice",
                label="Voice source",
                scale=1,
            )

        with gr.Group(visible=True) as preset_group:
            voice_preset = gr.Dropdown(
                choices=VOICE_PRESETS,
                value=VOICE_PRESETS[0],
                allow_custom_value=True,
                label="Voice preset",
                info="Select a preset or type any profile name from HuggingFace",
            )

        with gr.Group(visible=False) as custom_group:
            with gr.Row():
                voice_file = gr.Audio(
                    label="Reference audio (10-30 sec clip)",
                    type="filepath",
                    scale=1,
                )
                voice_script_text = gr.Textbox(
                    label="Transcript of the reference audio",
                    placeholder="Type the exact words spoken in your audio clip…",
                    lines=3,
                    scale=2,
                )

    def _toggle_voice(choice):
        return (
            gr.update(visible=(choice == "Preset Voice")),
            gr.update(visible=(choice == "Custom Voice")),
        )
    voice_type.change(_toggle_voice, voice_type, [preset_group, custom_group])

    # ── Advanced Settings ─────────────────────────────────────────
    with gr.Accordion("⚙️  Advanced Settings", open=False):
        with gr.Tabs():
            with gr.Tab("🔌 API Override"):
                gr.Markdown(
                    "*Override the LLM provider settings above. "
                    "Leave empty to use defaults.*",
                    elem_classes="openai-info",
                )
                api_base_url = gr.Textbox(
                    label="API Base URL",
                    placeholder="Auto-set by LLM Provider selection",
                )
                llm_model = gr.Textbox(
                    label="LLM Model",
                    placeholder="Auto-set by LLM Provider selection",
                )

            with gr.Tab("📥 Download"):
                quality = gr.Dropdown(
                    ["Low (360p)", "Medium (720p)", "High (best)"],
                    value="Medium (720p)",
                    label="Video quality",
                )
                with gr.Row():
                    start_time = gr.Textbox(
                        label="Start time",
                        placeholder="00:01:30 or 90",
                    )
                    end_time = gr.Textbox(
                        label="End time",
                        placeholder="00:05:00 or 300",
                    )

            with gr.Tab("📝 Transcription"):
                transcribe_method = gr.Dropdown(
                    list(METHOD_MAP.keys()),
                    value="Faster Whisper (local GPU)",
                    label="Transcription method",
                    info="Cloud = needs OpenAI key  •  Local = faster, requires GPU",
                )
                whisper_model = gr.Textbox(
                    label="Model override",
                    placeholder="whisper-1 (cloud) / large-v3 (local)",
                )

            with gr.Tab("🌐 Translation"):
                source_language = gr.Dropdown(
                    ["Auto-detect"] + LANGUAGES,
                    value="Auto-detect",
                    label="Source language",
                )
                with gr.Row():
                    words_per_second = gr.Slider(
                        1.0, 4.0, value=2.0, step=0.1,
                        label="Words per second",
                    )
                    duration_budget = gr.Slider(
                        0.5, 1.0, value=0.80, step=0.05,
                        label="Duration budget",
                    )
                translate_technical = gr.Checkbox(
                    label="Translate technical terms",
                    value=False,
                )

            with gr.Tab("🗣️ TTS"):
                tts_engine = gr.Dropdown(
                    ["Qwen", "Chatterbox"],
                    value="Qwen",
                    label="TTS engine",
                )
                tts_dtype = gr.Dropdown(
                    ["bfloat16", "float16", "float32"],
                    value="bfloat16",
                    label="Model precision",
                )
                with gr.Group(visible=False) as chatterbox_group:
                    with gr.Row():
                        chatterbox_exaggeration = gr.Slider(
                            0.0, 1.0, value=0.5, step=0.05,
                            label="Exaggeration",
                        )
                        chatterbox_cfg = gr.Slider(
                            0.0, 1.0, value=0.5, step=0.05,
                            label="CFG weight",
                        )

                def _toggle_cb(engine):
                    return gr.update(visible=(engine == "Chatterbox"))
                tts_engine.change(_toggle_cb, tts_engine, chatterbox_group)

            with gr.Tab("🔊 Audio"):
                tempo_mode = gr.Dropdown(
                    ["Auto", "Off", "Dynamic", "Fixed"],
                    value="Auto",
                    label="Tempo mode",
                )
                max_tempo = gr.Slider(
                    1.0, 2.0, value=1.3, step=0.05,
                    label="Max tempo",
                )
                with gr.Row():
                    loudness_match = gr.Checkbox(
                        label="Match original loudness",
                        value=True,
                    )
                    mix_background = gr.Checkbox(
                        label="Mix background audio",
                        value=True,
                    )
                background_volume = gr.Slider(
                    0.0, 1.0, value=0.15, step=0.05,
                    label="Background volume",
                )

            with gr.Tab("📦 Output"):
                output_type = gr.Dropdown(
                    ["Audio", "Video"],
                    value="Audio",
                    label="Output type",
                    info="Audio = dubbed WAV  •  Video = mux into source",
                )
                force_reset = gr.Checkbox(
                    label="Force reset (discard cache, re-run all stages)",
                    value=False,
                )

    gr.HTML('<hr class="divider">')

    # ── Run Button ────────────────────────────────────────────────
    run_btn = gr.Button(
        "🎬  Start Dubbing",
        variant="primary",
        size="lg",
        elem_classes="run-btn",
    )

    # ── Status & Logs ─────────────────────────────────────────────
    status = gr.Textbox(
        label="Status",
        interactive=False,
    )
    with gr.Accordion("📋 Pipeline Log", open=False):
        logs = gr.Textbox(
            label="Log output",
            lines=15,
            max_lines=40,
            interactive=False,
            elem_classes="log-box",
        )

    # ── Results ───────────────────────────────────────────────────
    gr.Markdown("#### 📦  RESULTS", elem_classes="section-title")
    with gr.Group(elem_classes="results-card"):
        with gr.Row():
            audio_output = gr.Audio(label="Dubbed Audio", type="filepath")
            video_output = gr.Video(label="Dubbed Video")

    # ── LLM provider toggle ───────────────────────────────────────
    # Auto-switches transcription method when LLM provider changes
    def _on_llm_provider_change(choice):
        is_ollama = (choice == "Ollama (Local — Free)")
        return (
            gr.update(visible=is_ollama),        # ollama_group
            gr.update(visible=not is_ollama),     # openai_group
            gr.update(value="Faster Whisper (local GPU)" if is_ollama
                      else "OpenAI Whisper (cloud)"),  # transcribe_method
        )
    llm_provider.change(
        _on_llm_provider_change, llm_provider,
        [ollama_group, openai_group, transcribe_method],
    )

    # ── Wire everything ───────────────────────────────────────────
    run_btn.click(
        fn=run_dubbing,
        inputs=[
            source_type, url_input, file_input,
            cookies_text,
            target_language, voice_type, voice_preset, voice_file, voice_script_text,
            llm_provider, ollama_model, openai_key,
            api_base_url, llm_model,
            quality, start_time, end_time,
            transcribe_method, whisper_model,
            source_language, words_per_second, duration_budget, translate_technical,
            tts_engine, tts_dtype, chatterbox_exaggeration, chatterbox_cfg,
            tempo_mode, max_tempo, loudness_match, mix_background, background_volume,
            output_type, force_reset,
        ],
        outputs=[status, logs, audio_output, video_output],
    )


# ═══════════════════════════════════════════════════════════════════════
#  Launch
# ═══════════════════════════════════════════════════════════════════════

app.launch(
    share=True,
    debug=True,
    show_error=True,
)
